In [ ]:
from src.dataset import AirSketchDataset, build_dataloaders
import yaml, json

with open("configs/default.yaml") as f:
    config = yaml.safe_load(f)

with open("data/splits/freihand_splits.json") as f:
    splits = json.load(f)

# Use a small subset for local exploration
SUBSET = 5000

train_ds = AirSketchDataset(
    landmarks_path = "data/processed/freihand/landmarks.npy",
    split_indices  = splits["train"][:SUBSET],
    augment        = False,  # off for visualization
)
print(f"Windows: {len(train_ds):,}")
print(f"Gesture distribution: {train_ds.gesture_distribution()}")
print(f"Class weights: {train_ds.get_class_weights()}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (0,9),(9,10),(10,11),(11,12),
    (0,13),(13,14),(14,15),(15,16),
    (0,17),(17,18),(18,19),(19,20),
    (5,9),(9,13),(13,17),
]
INDEX_FINGERTIP = 8

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle(
    "4 sample windows — each row = 4 evenly-spaced frames from one window\n"
    "Blue skeleton = landmarks, red dot = index fingertip, "
    "green cross = target position at T+1",
    fontsize=10,
)

for row, sample_idx in enumerate([0, 10, 50, 200]):
    sample   = train_ds[sample_idx]
    seq      = sample["sequence"].numpy().reshape(16, 21, 2)  # (T, 21, 2)
    target   = sample["target"].numpy()                        # (2,)
    gesture  = sample["gesture"].item()
    label    = "draw" if gesture == 1 else "idle"

    frame_indices = [0, 5, 10, 15]   # 4 evenly-spaced frames from the window
    for col, fi in enumerate(frame_indices):
        ax  = axes[row, col]
        uv  = seq[fi]   # (21, 2)

        # Draw skeleton
        for (a, b) in HAND_CONNECTIONS:
            ax.plot([uv[a,0], uv[b,0]], [uv[a,1], uv[b,1]],
                    "b-", linewidth=0.8, alpha=0.6)

        # Draw all landmarks
        ax.scatter(uv[:, 0], uv[:, 1], s=15, c="white",
                   edgecolors="steelblue", linewidths=0.5, zorder=5)

        # Highlight index fingertip
        ax.scatter(uv[INDEX_FINGERTIP, 0], uv[INDEX_FINGERTIP, 1],
                   s=50, c="red", zorder=6)

        # Draw target on the last frame column only
        if col == 3:
            ax.scatter(target[0], target[1],
                       s=80, c="lime", marker="+", linewidths=2, zorder=7,
                       label="target T+1")
            ax.legend(fontsize=7, loc="upper right")

        ax.set_xlim(0, 1)
        ax.set_ylim(1, 0)
        ax.set_aspect("equal")
        ax.axis("off")

        if col == 0:
            ax.set_ylabel(f"sample {sample_idx}\n({label})", fontsize=8)
        if row == 0:
            ax.set_title(f"frame {fi}", fontsize=8)

plt.tight_layout()
plt.savefig("report/figures/dataset_windows.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
from src.dataset import AirSketchDataset

train_aug = AirSketchDataset(
    landmarks_path = "data/processed/freihand/landmarks.npy",
    split_indices  = splits["train"][:SUBSET],
    augment        = True,
    flip_prob      = 1.0,    # force flip for visualization
    noise_sigma    = 0.0,    # disable noise so flip is clearly visible
)
train_no_aug = AirSketchDataset(
    landmarks_path = "data/processed/freihand/landmarks.npy",
    split_indices  = splits["train"][:SUBSET],
    augment        = False,
)

idx  = 5
orig = train_no_aug[idx]["sequence"].numpy().reshape(16, 21, 2)[0]
flip = train_aug[idx]["sequence"].numpy().reshape(16, 21, 2)[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(8, 4))
for ax, uv, title in [(ax1, orig, "Original"), (ax2, flip, "Flipped (x → 1-x)")]:
    for (a, b) in HAND_CONNECTIONS:
        ax.plot([uv[a,0], uv[b,0]], [uv[a,1], uv[b,1]], "b-", lw=0.8, alpha=0.6)
    ax.scatter(uv[:, 0], uv[:, 1], s=20, c="white", edgecolors="steelblue", lw=0.5)
    ax.scatter(uv[INDEX_FINGERTIP, 0], uv[INDEX_FINGERTIP, 1], s=60, c="red", zorder=5)
    ax.set_xlim(0, 1); ax.set_ylim(1, 0); ax.set_aspect("equal")
    ax.set_title(title, fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.savefig("report/figures/augmentation_flip.png", dpi=150, bbox_inches="tight")
plt.show()

# Numeric check: x coords should be mirrored
np.testing.assert_allclose(flip[:, 0], 1.0 - orig[:, 0], atol=1e-6)
print("✓ Flip verified: x_flipped == 1 - x_original")

In [ ]:
import matplotlib.pyplot as plt

dist = train_ds.gesture_distribution()
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["idle (0)", "draw (1)"], [dist["idle"], dist["draw"]],
       color=["#4A90D9", "#E84040"], edgecolor="none")
ax.set_title(f"Gesture label distribution — train split\n"
             f"(draw fraction: {dist['draw_frac']*100:.1f}%)", fontsize=10)
ax.set_ylabel("Window count")
plt.tight_layout()
plt.savefig("report/figures/gesture_distribution.png", dpi=150, bbox_inches="tight")
plt.show()